In [0]:
from dbr_py_utils.encryption_utils import AESEncryptor
from dbr_py_utils.aux_functions import df_string_field_curation, table_record_count_audit
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
%run ./Initial

In [0]:
expedia_silver_table = "silver.expedia_processed"
hotel_weather_silver_table = "silver.hotel_weather_processed"

In [0]:
expedia_silver_df = spark.read.table(expedia_silver_table)
hotel_weather_silver_df = spark.read.table(hotel_weather_silver_table)
logger.info(f"Read tables {expedia_silver_table} and {hotel_weather_silver_table}")

In [0]:
aes_key = dbutils.secrets.get("key-vault", "aes-encryption-key")
hotel_weather_encryptor = AESEncryptor(key=aes_key, attribute_list=["name", "address"])
hotel_weather_silver_df = hotel_weather_silver_df.transform(lambda df: hotel_weather_encryptor.aesDecrypt(df))
logger.info(f"Decrypted {hotel_weather_silver_table}")

### Top 10 Hotels with Maximum Absolute Temperature Difference by Month

In [0]:
top_10_hotel_temperature_difference_df = (
    hotel_weather_silver_df
    .withColumn("weather_year_month", F.concat(F.year("weather_date"), F.lit("-"), F.month("weather_date")))
    .groupBy(["id", "name", "weather_year_month"])
    .agg(
        F.min(F.col("average_temperature_celsius")).alias("min_temp_celsius"),
        F.max(F.col("average_temperature_celsius")).alias("max_temp_celsius")
    )
    .select(
        F.col("id").alias("hotel_id"),
        F.col("name").alias("hotel_name"),
        F.col("weather_year_month").alias("weather_year_month"),
        (F.col("max_temp_celsius") - F.col("min_temp_celsius")).alias("temperature_difference")
    )
    .withColumn("insert_timestamp_utc", F.lit(job_task_timestamp_utc).cast("timestamp"))
    .orderBy(F.col("temperature_difference").desc())
    .limit(10)
)

In [0]:
top_10_hotel_temperature_difference_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"gold.top_10_temp_diff_monthly")

logger.info(f"Persisted with overwrite processed top_10_hotel_temperature_difference_df to gold.top_10_temp_diff_monthly")
table_record_count_audit(spark, "gold.top_10_temp_diff_monthly", job_task_timestamp_utc)

### Top 10 Most Visited Hotels per Month

In [0]:
expedia_month_checkin_df = (
    expedia_silver_df
    .withColumn("checkin_year_month", F.concat(F.year("srch_ci"), F.lit("-"), F.month("srch_ci")))
    .groupBy(["hotel_id", "checkin_year_month"])
    .agg(
        F.sum(F.col("srch_adults_cnt")).alias("total_adults"),
        F.sum(F.col("srch_children_cnt")).alias("total_children")
    )
)

expedia_month_checkout_df = (
    expedia_silver_df
    .withColumn("checkout_year_month", F.concat(F.year("srch_co"), F.lit("-"), F.month("srch_co")))
    .groupBy(["hotel_id", "checkout_year_month"])
    .agg(
        F.sum(F.col("srch_adults_cnt")).alias("total_adults"),
        F.sum(F.col("srch_children_cnt")).alias("total_children")
    )
)

top_10_hotel_ids_month_visit_df = (
    expedia_month_checkin_df.alias("in")
    .join(
        other=expedia_month_checkout_df.alias("out"), 
        on = (F.col("in.hotel_id") == F.col("out.hotel_id")) & (F.col("in.checkin_year_month") == F.col("out.checkout_year_month")), 
        how = "full"
    )
    .select(
        F.coalesce(F.col("in.hotel_id"), F.col("out.hotel_id")).alias("hotel_id"),
        F.coalesce(F.col("in.checkin_year_month"), F.col("out.checkout_year_month")).alias("year_month"),
        (F.ifnull(F.col("in.total_adults"), F.lit(0))
        + F.ifnull(F.col("in.total_children"), F.lit(0))
        + F.ifnull(F.col("out.total_adults"), F.lit(0))
        + F.ifnull(F.col("out.total_children"), F.lit(0))).alias("total_visitors")
    )
    .orderBy(F.col("total_visitors").desc())
    .limit(10)
)

hotel_name_id_map_df = hotel_weather_silver_df.selectExpr("id as hotel_id", "name as hotel_name").distinct()

top_10_hotel_month_visit_df = top_10_hotel_ids_month_visit_df.alias("top").join(
    other=hotel_name_id_map_df.alias("map"), 
    on=F.col("top.hotel_id") == F.col("map.hotel_id"), 
    how="left"
).select(
    "top.hotel_id",
    "map.hotel_name",
    "top.year_month",
    "top.total_visitors"
).withColumn("insert_timestamp_utc", F.lit(job_task_timestamp_utc).cast("timestamp"))

In [0]:
top_10_hotel_month_visit_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"gold.top_10_busiest_hotels_monthly")

logger.info(f"Persisted with overwrite processed top_10_hotel_month_visit_df to gold.top_10_busiest_hotels_monthly")
table_record_count_audit(spark, "gold.top_10_busiest_hotels_monthly", job_task_timestamp_utc)

### Weather Trend for Extended Stays (more than 7 days)

In [0]:
expedia_silver_g7_filtered_df = expedia_silver_df.filter(F.datediff(F.col("srch_co"), F.col("srch_ci")) > 7)

hotel_stay_temp_history_df = (
    expedia_silver_g7_filtered_df
    .alias("exp")
    .join(
        other=hotel_weather_silver_df.alias("htl"), 
        on=((F.col("exp.hotel_id") == F.col("htl.id"))
        & (F.col("htl.weather_date").between(F.col("exp.srch_ci"), F.col("exp.srch_co")))),
        how="inner"
    )
    .selectExpr("exp.hotel_id", "htl.name as hotel_name", "exp.srch_ci", "exp.srch_co", "htl.average_temperature_celsius", "htl.weather_date")
    .withColumn("rnb_closest_to_checkin", F.row_number().over(Window.partitionBy("hotel_id", "srch_ci", "srch_co").orderBy(F.col("weather_date").asc())))
    .withColumn("rnb_closest_to_checkout", F.row_number().over(Window.partitionBy("hotel_id", "srch_ci", "srch_co").orderBy(F.col("weather_date").desc())))
)

hotel_stay_temp_stats_df = (
    hotel_stay_temp_history_df
    .groupBy(["hotel_id", "hotel_name", "srch_ci", "srch_co"])
    .agg(
        F.avg(F.col("average_temperature_celsius")).alias("average_stay_temperature_celsius"),
        F.max(F.when(F.col("rnb_closest_to_checkin") == 1, F.col("average_temperature_celsius"))).alias("closest_to_checkin_temperature_celsius"),
        F.max(F.when(F.col("rnb_closest_to_checkout") == 1, F.col("average_temperature_celsius"))).alias("closest_to_checkout_temperature_celsius")
    )
    .withColumn("insert_timestamp_utc", F.lit(job_task_timestamp_utc).cast("timestamp"))
)

In [0]:
hotel_stay_temp_stats_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"gold.weather_trend_extended_stay")

logger.info(f"Persisted with overwrite processed hotel_stay_temp_stats_df to gold.weather_trend_extended_stay")
table_record_count_audit(spark, "gold.weather_trend_extended_stay", job_task_timestamp_utc)